# Cross-Dataset Evaluation: PGVIB Variants — Aggregator

Merges the two split notebooks' CSV outputs and produces:

1. **Cross-dataset EER table** — mean ± std across seeds, per condition × dataset.
2. **Per-group VIB vs Monolithic deltas** — does PGVIB help on each held-out set?
3. **Aggregate verdict** — single-line decision rule for the thesis chapter.
4. **Visualizations** — per-dataset bar chart and a condition × dataset heatmap.

**Inputs:**
- `MyDrive/Jakob/Experiments/cross_dataset_eval_pgvib_results_part_a.csv` (2021 LA + 2021 DF)
- `MyDrive/Jakob/Experiments/cross_dataset_eval_pgvib_results_part_b.csv` (WaveFake + CodecFake+)

**Outputs:**
- `MyDrive/Jakob/Experiments/cross_dataset_eval_pgvib_summary.csv` — combined long table.
- `MyDrive/Jakob/Experiments/cross_dataset_eval_pgvib_pivot.csv` — condition × dataset pivot (for thesis).
- Inline figures: `cross_dataset_eer_bars.png`, `cross_dataset_eer_heatmap.png`.

Run this **after** both `cross_dataset_eval_pgvib_part_a.ipynb` and `_part_b.ipynb`
have finished writing their CSVs.


## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_EXP_BASE = '/content/drive/MyDrive/Jakob/Experiments'
PART_A_CSV = os.path.join(DRIVE_EXP_BASE, 'cross_dataset_eval_pgvib_results_part_a.csv')
PART_B_CSV = os.path.join(DRIVE_EXP_BASE, 'cross_dataset_eval_pgvib_results_part_b.csv')

for p in [PART_A_CSV, PART_B_CSV]:
    print(('OK   ' if os.path.exists(p) else 'MISS ') + p)


## 2. Load + concatenate both halves

In [ ]:
import pandas as pd

frames = []
for label, path in [('part_a', PART_A_CSV), ('part_b', PART_B_CSV)]:
    if not os.path.exists(path):
        print(f'[WARN] {label}: {path} missing — skipping.')
        continue
    df = pd.read_csv(path)
    df['source_part'] = label
    frames.append(df)

assert frames, 'No input CSVs found — run part_a and/or part_b first.'
df = pd.concat(frames, ignore_index=True)

# Sanity: no dataset should appear in both parts.
overlap = (
    df.groupby('dataset')['source_part']
      .nunique()
      .pipe(lambda s: s[s > 1])
)
if not overlap.empty:
    print(f'[WARN] Datasets present in both parts (will use both rows): {list(overlap.index)}')

print(f'Loaded {len(df)} rows; datasets: {sorted(df["dataset"].unique())}')
df.head()


## 3. Combined long table

One row per (variant, dataset). Saved to `cross_dataset_eval_pgvib_summary.csv`.


In [ ]:
SUMMARY_CSV = os.path.join(DRIVE_EXP_BASE, 'cross_dataset_eval_pgvib_summary.csv')
df.sort_values(['model_class', 'backbone', 'seed', 'dataset']).to_csv(SUMMARY_CSV, index=False)
print(f'Wrote {SUMMARY_CSV}  ({len(df)} rows)')


## 4. Pivot: condition × dataset (mean ± std across seeds)

A *condition* is `(model_class, backbone, model_kwargs)` — collapses seeds.
We label conditions with a short tag derived from the variant name prefix
(`monolithic_vib`, `pergroup_vib`, `g0_only_vib`, `drop_vib_g0`, `g0_only_large`,
`g0_only_xlsr`).


In [ ]:
import re

def condition_of(variant):
    # Strip trailing seed token: e.g. 'pergroup_vib_s42' -> 'pergroup_vib'.
    return re.sub(r'_s\d+$', '', variant)

df['condition'] = df['variant'].map(condition_of)

# mean / std EER per (condition, backbone, dataset)
g = df.groupby(['condition', 'backbone', 'dataset'])
agg = g['eer'].agg(['mean', 'std', 'count']).reset_index()
agg['cell'] = agg.apply(
    lambda r: f"{r['mean']:.2f}" if r['count'] == 1 or pd.isna(r['std'])
              else f"{r['mean']:.2f}±{r['std']:.2f}",
    axis=1,
)

pivot = agg.pivot_table(
    index=['condition', 'backbone'],
    columns='dataset',
    values='cell',
    aggfunc='first',
).reset_index()

# Order columns: identifier columns first, then datasets in canonical order.
DATASET_ORDER = ['2021_LA', '2021_DF', 'WaveFake', 'CodecFake']
ordered_cols = ['condition', 'backbone'] + [d for d in DATASET_ORDER if d in pivot.columns]
pivot = pivot[ordered_cols]

PIVOT_CSV = os.path.join(DRIVE_EXP_BASE, 'cross_dataset_eval_pgvib_pivot.csv')
pivot.to_csv(PIVOT_CSV, index=False)
print(f'Wrote {PIVOT_CSV}')
pivot


## 5. Per-group VIB vs Monolithic — delta per dataset

For each dataset, compare the mean EER of `pergroup_vib` against `monolithic_vib`
(both on `wavlm-base-plus`, the matched backbone). Negative delta = PGVIB wins.


In [ ]:
def mean_eer(condition, backbone, dataset):
    sub = df[(df['condition'] == condition) &
             (df['backbone'] == backbone) &
             (df['dataset'] == dataset)]
    if sub.empty:
        return None
    return sub['eer'].mean(), sub['eer'].std() if len(sub) > 1 else 0.0, len(sub)


backbone = 'wavlm-base-plus'
rows = []
for ds in DATASET_ORDER:
    if ds not in df['dataset'].unique():
        continue
    pg = mean_eer('pergroup_vib', backbone, ds)
    mo = mean_eer('monolithic_vib', backbone, ds)
    if pg is None or mo is None:
        rows.append({'dataset': ds, 'pergroup_eer': None, 'monolithic_eer': None,
                     'delta': None, 'verdict': 'incomplete'})
        continue
    pg_mean, pg_std, pg_n = pg
    mo_mean, mo_std, mo_n = mo
    delta = pg_mean - mo_mean
    rows.append({
        'dataset': ds,
        'pergroup_eer':  f'{pg_mean:.2f}±{pg_std:.2f} (n={pg_n})',
        'monolithic_eer': f'{mo_mean:.2f}±{mo_std:.2f} (n={mo_n})',
        'delta_eer':      round(delta, 3),
        'verdict':        'pergroup wins' if delta < 0 else ('tie' if abs(delta) < 0.05 else 'monolithic wins'),
    })
delta_df = pd.DataFrame(rows)
delta_df


## 6. Aggregate verdict

A single decision-criterion cell. Default rule:

- **Ship per-group VIB** if its mean EER is lower than monolithic on the
  majority of held-out datasets *and* the average EER delta across datasets
  is negative.
- Otherwise, **ship monolithic VIB**.

Edit `MARGIN` if you want to require a stricter improvement.


In [ ]:
MARGIN = 0.0  # require pergroup to beat monolithic by at least this much (EER %)

valid = delta_df[delta_df['delta_eer'].notna()].copy()
if valid.empty:
    print('No comparable rows available — verdict deferred.')
else:
    n_wins = (valid['delta_eer'] < -MARGIN).sum()
    n_total = len(valid)
    avg_delta = valid['delta_eer'].mean()
    print(f'Datasets compared:        {n_total}')
    print(f'Per-group VIB wins on:    {n_wins}/{n_total}')
    print(f'Mean EER delta (PG - MO): {avg_delta:+.3f}  (negative = PG better)')
    decision = (
        'SHIP: per-group VIB'
        if (n_wins > n_total / 2 and avg_delta < -MARGIN)
        else 'SHIP: monolithic VIB'
    )
    print()
    print('=' * 50)
    print(decision)
    print('=' * 50)


## 7. Visualizations

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

backbone = 'wavlm-base-plus'
conditions_to_plot = ['monolithic_vib', 'pergroup_vib', 'g0_only_vib', 'drop_vib_g0']
datasets_present = [d for d in DATASET_ORDER if d in df['dataset'].unique()]

x = np.arange(len(datasets_present))
width = 0.8 / max(1, len(conditions_to_plot))

fig, ax = plt.subplots(figsize=(9, 4.5))
for i, cond in enumerate(conditions_to_plot):
    means, stds = [], []
    for ds in datasets_present:
        r = mean_eer(cond, backbone, ds)
        if r is None:
            means.append(np.nan); stds.append(0)
        else:
            m, s, _ = r
            means.append(m); stds.append(s)
    ax.bar(x + i * width, means, width, yerr=stds, label=cond, capsize=3)

ax.set_xticks(x + width * (len(conditions_to_plot) - 1) / 2)
ax.set_xticklabels(datasets_present)
ax.set_ylabel('EER (%) — lower is better')
ax.set_title(f'Cross-dataset EER by condition ({backbone})')
ax.legend(loc='upper left', fontsize=9)
ax.grid(axis='y', alpha=0.3)
fig.tight_layout()
out_png = os.path.join(DRIVE_EXP_BASE, 'cross_dataset_eer_bars.png')
fig.savefig(out_png, dpi=150)
print(f'Saved {out_png}')
plt.show()


In [ ]:
# Heatmap: condition (rows) × dataset (cols), cell = mean EER.
import numpy as np

heatmap_conditions = ['monolithic_vib', 'pergroup_vib', 'g0_only_vib', 'drop_vib_g0',
                      'g0_only_large', 'g0_only_xlsr']
matrix = []
backbones_for_row = []
labels = []
for cond in heatmap_conditions:
    sub_b = df[df['condition'] == cond]['backbone'].unique()
    for bb in sub_b:
        row = []
        for ds in datasets_present:
            r = mean_eer(cond, bb, ds)
            row.append(np.nan if r is None else r[0])
        matrix.append(row)
        labels.append(f'{cond} / {bb}')

matrix = np.array(matrix, dtype=float)

fig, ax = plt.subplots(figsize=(7, 0.5 + 0.45 * len(labels)))
im = ax.imshow(matrix, aspect='auto', cmap='RdYlGn_r')
ax.set_xticks(range(len(datasets_present)))
ax.set_xticklabels(datasets_present, rotation=20, ha='right')
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=9)
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        v = matrix[i, j]
        if not np.isnan(v):
            ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                    color=('white' if v > np.nanmean(matrix) else 'black'),
                    fontsize=8)
plt.colorbar(im, ax=ax, label='EER (%)')
ax.set_title('Cross-dataset EER heatmap (mean across seeds)')
fig.tight_layout()
out_png = os.path.join(DRIVE_EXP_BASE, 'cross_dataset_eer_heatmap.png')
fig.savefig(out_png, dpi=150)
print(f'Saved {out_png}')
plt.show()


## 8. Notes for the report

- The pivot table (`cross_dataset_eval_pgvib_pivot.csv`) drops directly into the
  cross-dataset section of the thesis — copy-paste into the LaTeX table.
- The `delta_df` table above is the source for the "PGVIB vs monolithic" sentence
  in the verdict paragraph.
- If per-group VIB wins on a *minority* of datasets but with a large mean delta,
  reconsider `MARGIN` and document the choice.
